# 기법별 Ablation 실험 (11개, Colab A100)

**목적**: 카테고리 묶음이 아닌 **개별 기법별 기여도** 측정  
**환경**: Colab A100  
**공통 조건**: yolov8n.pt / epochs=15 / imgsz=640 / batch=16 / seed=42

## 실험 11개 목록

| # | exp_name | 기법 | p |
|---|---|---|---|
| B1 | exp_b1_degrees | degrees=10.0 | - |
| B2 | exp_b2_shear | shear=2.0 | - |
| B3 | exp_b3_perspective | perspective=0.0005 | - |
| C1 | exp_c1_gaussian_blur | GaussianBlur(blur_limit=7) | 0.3 |
| C2 | exp_c2_motion_blur | MotionBlur(blur_limit=7) | 0.3 |
| C3 | exp_c3_median_blur | MedianBlur(blur_limit=7) | 0.3 |
| D1 | exp_d1_gauss_noise | GaussNoise | 0.2 |
| D2 | exp_d2_iso_noise | ISONoise | 0.2 |
| E1 | exp_e1_clahe | CLAHE(clip=4.0) | 0.5 |
| E2 | exp_e2_rbc | RandomBrightnessContrast | 0.5 |
| F | exp_f_compression | ImageCompression(50-95) | 0.3 |

Baseline(A)은 지우님 JSON 재활용 → 실험 대상 아님.

## 사용법
1. 셀 1에서 본인 Drive 경로만 수정
2. 위에서 아래로 순차 실행
3. 중간에 세션 끊기면 이 노트북 처음부터 다시 실행 → 이미 완료된 실험은 자동 스킵하고 남은 것부터 재개

## 셀 1 — 설정 (경로 확인)

In [ ]:
AUTHOR = '승원'

# Drive 경로
DATASET_ZIP = '/content/drive/MyDrive/dataset_yolo.zip'
EVAL_PKG_PATH = '/content/drive/MyDrive/eval_package.zip'

# Drive 백업 위치 (실험 결과 전부 여기로)
BACKUP_ROOT = '/content/drive/MyDrive/pill_detection/results'
RESULTS_JSON_DIR = '/content/drive/MyDrive/pill_detection/results/_json_collection'

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
assert Path(DATASET_ZIP).exists(), f'데이터 zip 없음: {DATASET_ZIP}'
assert Path(EVAL_PKG_PATH).exists(), f'평가 패키지 없음: {EVAL_PKG_PATH}'

Path(BACKUP_ROOT).mkdir(parents=True, exist_ok=True)
Path(RESULTS_JSON_DIR).mkdir(parents=True, exist_ok=True)

!nvidia-smi | head -20

## 셀 2 — 데이터 압축 해제 (한 번만)

In [ ]:
import os

if not os.path.exists('/content/dataset_yolo/pill.yaml'):
    !unzip -l {DATASET_ZIP} | head -10
    !rm -rf /content/dataset_yolo
    !mkdir -p /content/dataset_yolo
    !unzip -q {DATASET_ZIP} -d /content/dataset_yolo/

    # 이중 래핑 자동 정리
    if os.path.isdir('/content/dataset_yolo/dataset_yolo'):
        !mv /content/dataset_yolo/dataset_yolo/* /content/dataset_yolo/
        !rmdir /content/dataset_yolo/dataset_yolo

    !ls /content/dataset_yolo/
else:
    print('/content/dataset_yolo 이미 존재 — 해제 스킵')

PILL_YAML = '/content/dataset_yolo/pill.yaml'
assert Path(PILL_YAML).exists(), 'pill.yaml 없음'

## 셀 2-2 — pill.yaml 경로 패치 (Colab용)


In [ ]:
# pill.yaml path를 Colab 경로로 덮어쓰기
import yaml

with open(PILL_YAML, 'r', encoding='utf-8') as f:
    y = yaml.safe_load(f)

# 확인
print('수정 전:')
print(f'  path: {y.get("path")}')
print(f'  train: {y.get("train")}')
print(f'  val: {y.get("val")}')

# Colab 경로로 덮어쓰기
y['path'] = '/content/dataset_yolo'
y['train'] = 'images/train'
y['val'] = 'images/val'

with open(PILL_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(y, f, allow_unicode=True, sort_keys=False)

# 확인
print('\n수정 후:')
with open(PILL_YAML, 'r', encoding='utf-8') as f:
    print(f.read())

## 셀 3 — 패키지 확인

In [ ]:
!pip install -q ultralytics albumentations

import ultralytics, albumentations as A
print(f'ultralytics: {ultralytics.__version__}')
print(f'albumentations: {A.__version__}')

## 셀 4 — 실험 정의 (11개)

각 실험의 `train_extras`(ultralytics 파라미터)와 `alb_transforms`(Albumentations 리스트)

In [ ]:
import albumentations as A

# Albumentations 버전 호환 헬퍼
def make_gauss_noise(p):
    try:
        return A.GaussNoise(std_range=(0.04, 0.22), p=p)  # 2.x
    except TypeError:
        return A.GaussNoise(var_limit=(10.0, 50.0), p=p)  # 1.x

def make_image_compression(p):
    try:
        return A.ImageCompression(quality_range=(50, 95), compression_type='jpeg', p=p)
    except TypeError:
        return A.ImageCompression(quality_lower=50, quality_upper=95, p=p)

# ultralytics Albumentations 기본 6개 (건드리지 않는 부분)
def get_base_transforms():
    return [
        A.Blur(p=0.01),
        A.MedianBlur(p=0.01),
        A.ToGray(p=0.01),
        A.CLAHE(p=0.01),
        A.RandomBrightnessContrast(p=0.0),
        A.RandomGamma(p=0.0),
    ]

# 실험 11개 정의
EXPERIMENTS = [
    # B: Geometric (ultralytics train 파라미터)
    dict(name='exp_b1_degrees',         train_extras={'degrees': 10.0},       alb_extras=[]),
    dict(name='exp_b2_shear',           train_extras={'shear': 2.0},          alb_extras=[]),
    dict(name='exp_b3_perspective',     train_extras={'perspective': 0.0005}, alb_extras=[]),

    # C: Blur (Albumentations)
    dict(name='exp_c1_gaussian_blur',   train_extras={}, alb_extras=[A.GaussianBlur(blur_limit=7, p=0.3)]),
    dict(name='exp_c2_motion_blur',     train_extras={}, alb_extras=[A.MotionBlur(blur_limit=7, p=0.3)]),
    dict(name='exp_c3_median_blur',     train_extras={}, alb_extras=[A.MedianBlur(blur_limit=7, p=0.3)]),

    # D: Noise
    dict(name='exp_d1_gauss_noise',     train_extras={}, alb_extras=[make_gauss_noise(0.2)]),
    dict(name='exp_d2_iso_noise',       train_extras={}, alb_extras=[A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=0.2)]),

    # E: Color
    dict(name='exp_e1_clahe',           train_extras={}, alb_extras=[A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.5)]),
    dict(name='exp_e2_rbc',             train_extras={}, alb_extras=[A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5)]),

    # F: Compression
    dict(name='exp_f_compression',      train_extras={}, alb_extras=[make_image_compression(0.3)]),
]

print(f'총 {len(EXPERIMENTS)}개 실험 정의됨')
for e in EXPERIMENTS:
    print(f'  - {e["name"]}')

## 셀 5 — 평가셋 압축 해제 (한 번만)

In [ ]:
import zipfile, json

EVAL_DIR = Path('/content/_eval_workspace')
if not (EVAL_DIR / 'answer_key.json').exists():
    EVAL_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(EVAL_PKG_PATH, 'r') as zf:
        zf.extractall(EVAL_DIR)
    print('평가셋 해제 완료')
else:
    print('평가셋 이미 해제됨 — 스킵')

ANSWER_KEY_RAW = json.load(open(EVAL_DIR / 'answer_key.json', encoding='utf-8'))
TEST_IMG_DIR = EVAL_DIR / 'test_images'
assert len(ANSWER_KEY_RAW) == 33

## 셀 6 — 핵심 헬퍼 (몽키패치/학습/평가 함수)

In [ ]:
import sys, shutil, json
from pathlib import Path
from datetime import datetime
import numpy as np
from PIL import Image

# ---------- 몽키패치 ----------
def apply_monkey_patch(alb_extras):
    """sys.modules 초기화 + Albumentations.__init__ 교체"""
    mods = [k for k in sys.modules.keys() if k.startswith('ultralytics')]
    for m in mods:
        del sys.modules[m]

    from ultralytics.data.augment import Albumentations
    import albumentations as A_

    _original_init = Albumentations.__init__

    transforms_list = get_base_transforms() + alb_extras

    def _patched_init(self, p=1.0, transforms=None):
        _original_init(self, p=p, transforms=transforms_list)

    Albumentations.__init__ = _patched_init
    return Albumentations

# ---------- 학습 ----------
def run_training(exp_name, train_extras):
    from ultralytics import YOLO
    model = YOLO('yolov8n.pt')

    train_kwargs = dict(
        data=PILL_YAML,
        epochs=15,
        imgsz=640,
        batch=16,
        seed=42,
        project='/content/runs',
        name=exp_name,
        exist_ok=True,
    )
    train_kwargs.update(train_extras)

    model.train(**train_kwargs)
    return model

# ---------- 평가 ----------
def box_iou(b1, b2):
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    union = a1 + a2 - inter
    return inter / union if union > 0 else 0

def group_by_iou(dets, iou_thr=0.5):
    groups = []
    for det in sorted(dets, key=lambda x: -x['conf']):
        placed = False
        for g in groups:
            if box_iou(det['bbox'], g[0]['bbox']) >= iou_thr:
                g.append(det); placed = True; break
        if not placed:
            groups.append([det])
    return groups

def run_evaluation(model, exp_name):
    # model.names 기준으로 answer_key 동기화
    if isinstance(model.names, dict):
        name_to_idx = {v: k for k, v in model.names.items()}
    else:
        name_to_idx = {n: i for i, n in enumerate(model.names)}

    ANSWER_KEY = {}
    for fname, info in ANSWER_KEY_RAW.items():
        dl_name = info['dl_name']
        if dl_name not in name_to_idx:
            raise RuntimeError(f'{exp_name}: model.names에 없는 dl_name: {dl_name}')
        ANSWER_KEY[fname] = {
            'dl_name': dl_name,
            'class_idx': name_to_idx[dl_name],
            'dl_idx': info.get('dl_idx'),
        }

    idx_to_name = model.names if isinstance(model.names, dict) else {i: n for i, n in enumerate(model.names)}
    results = []

    for fn, info in ANSWER_KEY.items():
        img_path = TEST_IMG_DIR / fn
        W, H = Image.open(img_path).size
        pred = model.predict(source=str(img_path), conf=0.01, iou=0.7, verbose=False, imgsz=640)[0]

        dets = []
        if pred.boxes is not None and len(pred.boxes) > 0:
            for b in pred.boxes:
                xyxy = b.xyxy[0].cpu().numpy().tolist()
                dets.append({'bbox': xyxy, 'cls': int(b.cls[0].item()), 'conf': float(b.conf[0].item())})

        groups = group_by_iou(dets, 0.5)
        detections = []
        for g in groups:
            per_cls = {}
            for d in g:
                if d['cls'] not in per_cls or d['conf'] > per_cls[d['cls']]['conf']:
                    per_cls[d['cls']] = d
            ranked = sorted(per_cls.values(), key=lambda x: -x['conf'])[:3]
            detections.append({
                'bbox': [round(x, 1) for x in g[0]['bbox']],
                'top3_class_idx': [d['cls'] for d in ranked],
                'top3_class_name': [idx_to_name[d['cls']] for d in ranked],
                'top3_confidence': [round(d['conf'], 4) for d in ranked],
            })

        results.append({
            'filename': fn, 'img_width': W, 'img_height': H,
            'gt_dl_name': info['dl_name'], 'gt_class_idx': info['class_idx'],
            'num_detections': len(detections), 'detections': detections,
        })

    # 이름 기반 채점
    top1_hit = top3_hit = detected = 0
    for r in results:
        gt_name = r['gt_dl_name']
        if r['num_detections'] == 0: continue
        detected += 1
        top1_names = [d['top3_class_name'][0] for d in r['detections'] if d.get('top3_class_name')]
        top3_names = [n for d in r['detections'] for n in d.get('top3_class_name', [])]
        if gt_name in top1_names: top1_hit += 1
        if gt_name in top3_names: top3_hit += 1

    n = len(results)
    summary = {
        'exp_name': exp_name, 'author': AUTHOR,
        'timestamp': datetime.now().isoformat(),
        'eval_version': 'v2', 'num_images': n,
        'detection_rate': round(detected / n, 4),
        'top1_accuracy': round(top1_hit / n, 4),
        'top3_accuracy': round(top3_hit / n, 4),
        'results': results,
    }
    return summary

# ---------- Drive 백업 ----------
def backup_to_drive(exp_name, summary):
    # 결과 JSON
    json_path = Path(RESULTS_JSON_DIR) / f'{exp_name}_results.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    # 학습 결과 폴더 (weights, args.yaml 등)
    src = f'/content/runs/{exp_name}'
    dst = f'{BACKUP_ROOT}/{exp_name}'
    if Path(dst).exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'  Drive 백업 완료: {json_path.name}')

print('헬퍼 함수 정의 완료')

## 셀 7 — 11개 실험 순차 실행

이미 완료된(Drive에 JSON 있는) 실험은 자동 스킵.  
중간에 세션 끊겨도 이 셀 다시 돌리면 남은 것부터 재개.

In [ ]:
import time

for i, exp in enumerate(EXPERIMENTS, 1):
    exp_name = exp['name']
    json_path = Path(RESULTS_JSON_DIR) / f'{exp_name}_results.json'

    # 이미 완료된 실험 스킵
    if json_path.exists():
        with open(json_path, 'r', encoding='utf-8') as f:
            done = json.load(f)
        print(f'[{i}/{len(EXPERIMENTS)}] {exp_name} — 스킵 (Top-1 {done["top1_accuracy"]:.1%} / Top-3 {done["top3_accuracy"]:.1%})')
        continue

    print(f'\n{"="*60}')
    print(f'[{i}/{len(EXPERIMENTS)}] {exp_name} 시작')
    print(f'  train_extras: {exp["train_extras"]}')
    print(f'  alb_extras: {[type(t).__name__ for t in exp["alb_extras"]]}')
    print(f'{"="*60}')
    t0 = time.time()

    # 1) 몽키패치 (이전 실험 흔적 초기화 + 새 증강 주입)
    apply_monkey_patch(exp['alb_extras'])

    # 2) 학습
    model = run_training(exp_name, exp['train_extras'])

    # 3) 평가
    summary = run_evaluation(model, exp_name)

    # 4) Drive 백업
    backup_to_drive(exp_name, summary)

    elapsed = time.time() - t0
    print(f'\n  완료: Top-1 {summary["top1_accuracy"]:.1%} / Top-3 {summary["top3_accuracy"]:.1%} / {elapsed/60:.1f}분')

    # 메모리 정리
    del model
    import gc; gc.collect()
    import torch; torch.cuda.empty_cache()

print(f'\n\n{"="*60}')
print('전체 실험 완료')
print(f'{"="*60}')

## 셀 8 — 전체 결과 요약

In [ ]:
import json
from pathlib import Path

print(f'{"exp_name":30s} | Detection | Top-1  | Top-3')
print('-' * 70)

rows = []
for exp in EXPERIMENTS:
    json_path = Path(RESULTS_JSON_DIR) / f'{exp["name"]}_results.json'
    if not json_path.exists():
        print(f'{exp["name"]:30s} | 대기')
        continue
    with open(json_path, 'r', encoding='utf-8') as f:
        d = json.load(f)
    rows.append(d)
    print(f'{d["exp_name"]:30s} | {d["detection_rate"]:.3f}   | {d["top1_accuracy"]:.3f} | {d["top3_accuracy"]:.3f}')

print(f'\n총 {len(rows)}/{len(EXPERIMENTS)} 실험 완료')
print(f'결과 JSON 위치: {RESULTS_JSON_DIR}')
print(f'\n→ 이 폴더의 JSON들을 D:\\pill_detection\\eval\\results\\ 에 다운로드하고 compare_experiments.ipynb 돌리면 끝')